# lazymerge demo — DataFusion backend

Same mosaic as the conventions demo, but using the `datafusion=True` merge path.
Source arrays are organized in a zarr-datafusion-search store layout: a `/meta` group
with columnar metadata arrays, and source groups containing band subgroups with
multiscale overviews.

Instead of building an in-memory `ScanIndex` upfront, `merge(datafusion=True)` issues
per-chunk spatial SQL queries against the `/meta` group to find intersecting sources.
This scales to stores with many sources.

The `store` parameter in `merge()` accepts any zarr-compatible store (zarr Store,
Icechunk Session) or an obstore object store (LocalStore, S3Store). When
`datafusion=True`, the store must be obstore-compatible since `zarr-datafusion-search`
queries via obstore.

**Extra dependencies:**

```
uv pip install xarray matplotlib rasterix zarr-datafusion-search obstore
```

In [ ]:
import tempfile

import numpy as np
import xarray as xr
import zarr
from affine import Affine
from obstore.store import LocalStore
from rasterix import RasterIndex

from lazymerge.conventions import (
    ProjAttrs,
    SpatialAttrs,
    read_multiscales,
    read_spatial,
    write_proj,
    write_spatial,
)
from lazymerge.merge import merge
from lazymerge.sources import select_overview
from lazymerge.target import create_target

## 1. Create synthetic source arrays with DataFusion store layout

Four tiles arranged in a 2x2 grid. The left two tiles are in **UTM 18N** (EPSG:32618),
and the right two are in **UTM 17N** (EPSG:32617) — covering adjacent geographic areas
but stored in different coordinate reference systems.

Each source is a group containing a `data/` band subgroup with base (10m) and
overview (20m) arrays, plus the zarr multiscales convention.

We write to a local directory store so we can pass an obstore `LocalStore` to `merge()`.
DataFusion queries require an obstore-compatible store.

```
store_path/
  meta/               # columnar metadata for DataFusion queries
  utm18n_tile_a/
    data/
      multiscales      # convention attr
      0/               # base (10m)
      1/               # overview (20m)
  ...
```

In [ ]:
tmpdir = tempfile.mkdtemp()
store = zarr.storage.LocalStore(tmpdir)
root = zarr.open_group(store, mode="w")

sources = [
    # UTM 18N tiles (left side)
    {
        "name": "utm18n_tile_a",
        "crs": "EPSG:32618",
        "epsg": 32618,
        "bbox": (500000.0, 5990000.0, 505000.0, 5995000.0),
        "resolution": 10.0,
        "fill": 100.0,
    },
    {
        "name": "utm18n_tile_c",
        "crs": "EPSG:32618",
        "epsg": 32618,
        "bbox": (500000.0, 5995000.0, 505000.0, 6000000.0),
        "resolution": 10.0,
        "fill": 300.0,
    },
    # UTM 17N tiles (right side — same geographic area as UTM 18N
    # x=[503000, 508000], but expressed in UTM zone 17N coordinates)
    {
        "name": "utm17n_tile_b",
        "crs": "EPSG:32617",
        "epsg": 32617,
        "bbox": (895094.0, 6006920.0, 900511.0, 6012337.0),
        "resolution": 10.0,
        "fill": 200.0,
    },
    {
        "name": "utm17n_tile_d",
        "crs": "EPSG:32617",
        "epsg": 32617,
        "bbox": (894669.0, 6011912.0, 900086.0, 6017329.0),
        "resolution": 10.0,
        "fill": 400.0,
    },
]

for src in sources:
    xmin, ymin, xmax, ymax = src["bbox"]
    res = src["resolution"]
    width = int((xmax - xmin) / res)
    height = int((ymax - ymin) / res)

    scene_group = root.create_group(src["name"])
    band_group = scene_group.create_group("data")

    # Base resolution array (level 0)
    base = band_group.create_array("0", shape=(height, width), dtype="f4", chunks=(256, 256))
    base[:] = src["fill"]
    write_spatial(
        base,
        SpatialAttrs(
            dimensions=["y", "x"],
            transform=(res, 0.0, xmin, 0.0, -res, ymax),
            bbox=(xmin, ymin, xmax, ymax),
            shape=(height, width),
        ),
    )
    write_proj(base, ProjAttrs(code=src["crs"]))

    # 2x overview array (level 1)
    ovr_width = width // 2
    ovr_height = height // 2
    ovr_res = res * 2
    ovr = band_group.create_array("1", shape=(ovr_height, ovr_width), dtype="f4", chunks=(256, 256))
    ovr[:] = src["fill"]
    write_spatial(
        ovr,
        SpatialAttrs(
            dimensions=["y", "x"],
            transform=(ovr_res, 0.0, xmin, 0.0, -ovr_res, ymax),
            bbox=(xmin, ymin, xmax, ymax),
            shape=(ovr_height, ovr_width),
        ),
    )
    write_proj(ovr, ProjAttrs(code=src["crs"]))

    # Set multiscales convention on band group
    band_group.attrs["multiscales"] = {
        "layout": [
            {"asset": "0", "transform": {"scale": [1.0, 1.0], "translation": [0.0, 0.0]}},
            {
                "asset": "1",
                "derived_from": "0",
                "transform": {"scale": [2.0, 2.0], "translation": [0.5, 0.5]},
            },
        ]
    }

    print(f"{src['name']}: shape=({height}, {width}), crs={src['crs']}, fill={src['fill']}")

# Create obstore LocalStore pointing at the same directory for DataFusion queries
ob_store = LocalStore(tmpdir)
print(f"\nStore path: {tmpdir}")

## 2. Populate the `/meta` group

The `/meta` group holds columnar arrays that DataFusion queries against.
Each row corresponds to one source scene. By convention, bbox values are
stored in **EPSG:4326** so queries work in a single CRS.

In [3]:
import struct

from pyproj import Transformer


def bbox_to_wkb(xmin, ymin, xmax, ymax):
    """Encode a bounding box as a WKB Polygon (little-endian)."""
    points = [(xmin, ymin), (xmax, ymin), (xmax, ymax), (xmin, ymax), (xmin, ymin)]
    wkb = struct.pack("<BII", 1, 3, 1)  # LE, Polygon, 1 ring
    wkb += struct.pack("<I", 5)  # 5 points
    for x, y in points:
        wkb += struct.pack("<dd", x, y)
    return wkb


meta = root.create_group("meta")

ids = np.array([s["name"] for s in sources])
epsgs = np.array([s["epsg"] for s in sources], dtype="int64")

# Affine transform components: (a=res, b=0, c=xmin, d=0, e=-res, f=ymax)
t0 = np.array([s["resolution"] for s in sources])
t1 = np.zeros(len(sources))
t2 = np.array([s["bbox"][0] for s in sources])  # xmin
t3 = np.zeros(len(sources))
t4 = np.array([-s["resolution"] for s in sources])
t5 = np.array([s["bbox"][3] for s in sources])  # ymax

shape_x = np.array([int((s["bbox"][2] - s["bbox"][0]) / s["resolution"]) for s in sources], dtype="int64")
shape_y = np.array([int((s["bbox"][3] - s["bbox"][1]) / s["resolution"]) for s in sources], dtype="int64")

# Compute bboxes in EPSG:4326 and encode as WKB (DataFusion convention)
bbox_wkb = []
for s in sources:
    xmin, ymin, xmax, ymax = s["bbox"]
    transformer = Transformer.from_crs(s["crs"], "EPSG:4326", always_xy=True)
    xs = [xmin, xmin, xmax, xmax]
    ys = [ymin, ymax, ymin, ymax]
    tx, ty = transformer.transform(xs, ys)
    lon_min, lon_max = min(tx), max(tx)
    lat_min, lat_max = min(ty), max(ty)
    bbox_wkb.append(bbox_to_wkb(lon_min, lat_min, lon_max, lat_max))
    print(f"  {s['name']}: [{lon_min:.4f}, {lat_min:.4f}, {lon_max:.4f}, {lat_max:.4f}]")

bbox_wkb_arr = np.array(bbox_wkb, dtype=object)

# Use dtype='string' for variable-length UTF-8 (zarr VariableLengthUTF8)
id_arr = meta.create_array("id", shape=ids.shape, dtype="string")
id_arr[:] = ids
# Store bbox as variable-length bytes (WKB polygons in EPSG:4326)
bbox_arr = meta.create_array("bbox", shape=bbox_wkb_arr.shape, dtype="variable_length_bytes")
bbox_arr[:] = bbox_wkb_arr
meta.create_array("proj:epsg", data=epsgs)
meta.create_array("transform_0", data=t0)
meta.create_array("transform_1", data=t1)
meta.create_array("transform_2", data=t2)
meta.create_array("transform_3", data=t3)
meta.create_array("transform_4", data=t4)
meta.create_array("transform_5", data=t5)
meta.create_array("shape_x", data=shape_x)
meta.create_array("shape_y", data=shape_y)

print(f"\nPopulated /meta with {len(ids)} rows")

  utm18n_tile_a: [-75.0000, 54.0582, -74.9235, 54.1032]
  utm18n_tile_c: [-75.0000, 54.1031, -74.9234, 54.1481]
  utm17n_tile_b: [-74.9606, 54.0544, -74.8712, 54.1070]
  utm17n_tile_d: [-74.9606, 54.0993, -74.8710, 54.1519]

Populated /meta with 4 rows


/Users/seanharkins/projects/lazymerge/.venv/lib/python3.12/site-packages/zarr/core/dtype/npy/bytes.py:1143: UnstableSpecificationWarning: The data type (VariableLengthBytes()) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


## 3. Overview selection

Before running the full merge, let's demonstrate how overview selection works.
This part uses only `lazymerge` (no DataFusion dependency).

In [4]:
band_group = root["utm18n_tile_a/data"]
overviews = read_multiscales(band_group)
base_spatial = read_spatial(band_group["0"])
native_res = abs(base_spatial.transform[0])

print(f"Native resolution: {native_res}m")
print(f"Available overviews:")
for ov in overviews:
    print(f"  level '{ov.path}': {ov.resolution}m (scale={ov.scale})")

# At 10m target, no overview selected (use base)
sel_10 = select_overview(overviews, target_res=10.0, native_res=native_res)
print(f"\nTarget 10m -> selected: {sel_10} (use base)")

# At 20m target, overview 1 is selected
sel_20 = select_overview(overviews, target_res=20.0, native_res=native_res)
print(f"Target 20m -> selected: level '{sel_20.path}' @ {sel_20.resolution}m")

Native resolution: 10.0m
Available overviews:
  level '1': 20.0m (scale=(2.0, 2.0))

Target 10m -> selected: None (use base)
Target 20m -> selected: level '1' @ 20.0m


## 4. Merge with DataFusion

With `datafusion=True`, `merge()` does **not** need a pre-built `ScanIndex`.
Instead, each target chunk issues a DataFusion SQL query against the `/meta`
group to find intersecting sources by bbox.

The store must be obstore-compatible (e.g. `LocalStore`, `S3Store`) when using
the DataFusion path. For the convention path, any zarr-compatible store works
(zarr Store, Icechunk Session, or a string path).

```python
# No scan_store() call needed!
result_arr, result_spatial, result_proj = merge(
    source_index=None,       # no in-memory index
    target=target,
    target_spatial=spatial,
    target_proj=proj,
    store=ob_store,          # obstore LocalStore for DataFusion queries
    band="data",             # navigate into data/ band group
    datafusion=True,         # use DataFusion for source discovery
)
```

**Note:** The cell below requires `zarr-datafusion-search` and `obstore` to be installed.
Install with: `uv pip install zarr-datafusion-search obstore`

In [ ]:
target, spatial, proj = create_target(
    crs="EPSG:32618",
    bbox=(500000.0, 5990000.0, 508000.0, 6000000.0),
    resolution=10.0,
    chunk_size=(256, 256),
)
print(f"Target shape: {target.shape}")
print(f"Target chunks: {target.chunksize}")
print(f"Target CRS: {proj.code}")
print(f"Target bbox: {spatial.bbox}")

In [ ]:
# Requires zarr-datafusion-search and obstore
result_arr, result_spatial, result_proj = merge(
    source_index=None,
    target=target,
    target_spatial=spatial,
    target_proj=proj,
    store=ob_store,
    band="data",
    datafusion=True,
)

data = result_arr.compute()

# Build georeferenced coordinates using rasterix
affine = Affine(*result_spatial.transform)
da = xr.DataArray(data, dims=["y", "x"])

raster_idx = RasterIndex.from_transform(
    affine=affine,
    width=da.sizes["x"],
    height=da.sizes["y"],
    x_dim="x",
    y_dim="y",
    crs=result_proj.code,
)
coords = xr.Coordinates.from_xindex(raster_idx)
da = da.assign_coords(coords)
da = da.proj.assign_crs(spatial_ref=result_proj.code, allow_override=True)
da

## 5. Visualize

In [ ]:
da.plot(figsize=(12, 6), cmap="viridis", add_colorbar=True);

## 6. Overview selection at 20m

Merge at 20m to trigger the 2x overview. `merge()` automatically
selects the coarsest overview whose resolution is still <= the target.

In [ ]:
target_20m, spatial_20m, proj_20m = create_target(
    crs="EPSG:32618",
    bbox=(500000.0, 5990000.0, 508000.0, 6000000.0),
    resolution=20.0,
    chunk_size=(128, 128),
)

result_20m, result_spatial_20m, result_proj_20m = merge(
    source_index=None,
    target=target_20m,
    target_spatial=spatial_20m,
    target_proj=proj_20m,
    store=ob_store,
    band="data",
    datafusion=True,
)

data_20m = result_20m.compute()

affine_20m = Affine(*result_spatial_20m.transform)
da_20m = xr.DataArray(data_20m, dims=["y", "x"])

raster_idx_20m = RasterIndex.from_transform(
    affine=affine_20m,
    width=da_20m.sizes["x"],
    height=da_20m.sizes["y"],
    x_dim="x",
    y_dim="y",
    crs=result_proj_20m.code,
)
coords_20m = xr.Coordinates.from_xindex(raster_idx_20m)
da_20m = da_20m.assign_coords(coords_20m)
da_20m = da_20m.proj.assign_crs(spatial_ref=result_proj_20m.code, allow_override=True)

print(f"20m result shape: {data_20m.shape} (vs 10m: {data.shape})")
print(f"Unique values: {np.unique(data_20m[~np.isnan(data_20m)])}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

da.plot(ax=ax1, cmap="viridis", vmin=100, vmax=400, add_colorbar=False)
ax1.set_title(f"10m — base ({da.sizes['y']}x{da.sizes['x']})")
ax1.set_aspect("equal")

da_20m.plot(ax=ax2, cmap="viridis", vmin=100, vmax=400, add_colorbar=False)
ax2.set_title(f"20m — overview ({da_20m.sizes['y']}x{da_20m.sizes['x']})")
ax2.set_aspect("equal")

fig.suptitle("Overview selection: 10m uses base, 20m uses 2x overview", fontsize=14)
fig.tight_layout();

## 7. Summary

| Feature | Convention path | DataFusion path |
|---|---|---|
| Discovery | `scan_store(root)` builds in-memory index | Per-chunk SQL query against `/meta` |
| Scales to | Hundreds of sources | Millions of sources |
| Store type | Any zarr-compatible (zarr Store, Icechunk Session, str) | obstore-compatible (`LocalStore`, `S3Store`) |
| Merge call | `merge(source_index=index, store=zarr_store, ...)` | `merge(source_index=None, store=ob_store, datafusion=True, ...)` |
| Band navigation | `band="red"` | `band="red"` |
| Overview selection | Automatic via multiscales | Automatic via multiscales |